In [14]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "common.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common import PERIODS

PERIOD = "1990-2004"
START, END = PERIODS[PERIOD]
RESULTS = ROOT / "periods" / PERIOD / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

## Data

In [15]:
raw = pd.read_csv(ROOT / "data" / "raw" / "2026-09-MD.csv")
tcodes = pd.to_numeric(raw.iloc[0, 1:]).astype(int)

raw_panel = raw.iloc[1:].copy()
raw_panel.index = pd.to_datetime(raw_panel["sasdate"], format="%m/%d/%Y")
raw_panel = raw_panel.drop(columns="sasdate").apply(pd.to_numeric, errors="coerce")

def transform(s, c):
    with np.errstate(all="ignore"):
        if c == 1: return s
        if c == 2: return s.diff()
        if c == 3: return s.diff().diff()
        if c == 4: return np.log(s)
        if c == 5: return np.log(s).diff()
        if c == 6: return np.log(s).diff().diff()
        if c == 7: return (s / s.shift(1) - 1).diff()

# no outlier removal; first two rows are lost to differencing
tf_full = pd.DataFrame({v: transform(raw_panel[v], tcodes[v]) for v in raw_panel.columns}).iloc[2:]
tf_period = tf_full.loc[START:END]
print(tf_period.shape)   # (180, 126)

(180, 126)


In [16]:
audit = pd.DataFrame({
    "n_missing": tf_period.isna().sum(),
    "pct_missing": (100 * tf_period.isna().mean()).round(1),
    "first_obs": tf_period.apply(lambda s: s.first_valid_index()).dt.strftime("%Y-%m"),
    "last_obs": tf_period.apply(lambda s: s.last_valid_index()).dt.strftime("%Y-%m"),
})
print(audit[audit.n_missing > 0].sort_values("n_missing", ascending=False))
print("complete series:", int((audit.n_missing == 0).sum()), "of", tf_period.shape[1])

        n_missing  pct_missing first_obs last_obs
ACOGNO         26         14.4   1992-03  2004-12
complete series: 125 of 126


In [17]:
med = tf_period.median()
iqr = tf_period.quantile(0.75) - tf_period.quantile(0.25)
ratio = (tf_period - med).abs() / iqr.where(iqr > 0)
hits = ratio.stack()
flagged = hits[hits > 10].reset_index()
flagged.columns = ["date", "series", "x_over_IQR"]
flagged["date"] = flagged["date"].dt.strftime("%Y-%m")
print(flagged.round(2).to_string(index=False))

   date   series  x_over_IQR
2001-10     M1SL       12.96
2001-10 TOTRESNS       11.45


In [12]:
results = ROOT / "periods" / "1990-2004" / "results"
results.mkdir(parents=True, exist_ok=True)

print("series with NA, no outlier removal:", W.columns[W.isna().any()].tolist())

# Descriptive statistics (transformed series, 1990-2004)
stats = pd.DataFrame({
    "tcode": tcode, "n": W.count(), "mean": W.mean(), "sd": W.std(),
    "skew": W.skew(), "kurt": W.kurt() + 3,
})
stats.round(4).to_csv(results / "descriptive_stats.csv")
print("\nfattest tails:")
print(stats.sort_values("kurt", ascending=False).head(6).round(3))

# Correlation structure (pairwise, so ACOGNO is kept)
C = W.corr()
upper = C.where(np.triu(np.ones(C.shape), k=1).astype(bool)).stack()
top = upper.reindex(upper.abs().sort_values(ascending=False).index)
top.head(50).round(3).to_csv(results / "top_correlated_pairs.csv")
print("\nmost correlated pairs:")
print(top.head(8).round(3))
print("share of pairs with |corr| > 0.5:", round(float((upper.abs() > 0.5).mean()), 3))

# How much does 2001-10 matter for the correlations?
W_no = W.drop(index=pd.Timestamp("2001-10-01"))
for a, b in [("M1SL", "TOTRESNS"), ("TOTRESNS", "NONBORRES")]:
    print(a, b, "with 2001-10:", round(W[a].corr(W[b]), 3),
          "| without:", round(W_no[a].corr(W_no[b]), 3))

series with NA, no outlier removal: ['ACOGNO']

fattest tails:
                 tcode    n   mean     sd   skew    kurt
TOTRESNS             6  180 -0.000  0.069 -3.415  44.186
M1SL                 6  180 -0.000  0.009 -3.351  42.843
NONBORRES            7  180 -0.000  0.064 -2.615  37.141
DSERRG3M086SBEA      6  180 -0.000  0.002  2.043  25.255
W875RX1              5  180  0.002  0.007  1.139  21.311
RPI                  5  180  0.003  0.006  1.514  21.084

most correlated pairs:
TOTRESNS    NONBORRES     0.994
AAAFFM      BAAFFM        0.991
PERMIT      PERMITS       0.990
RPI         W875RX1       0.990
WPSFD49207  WPSFD49502    0.988
IPFPNSS     IPFINAL       0.980
T5YFFM      T10YFFM       0.980
HOUST       HOUSTS        0.977
dtype: float64
share of pairs with |corr| > 0.5: 0.021
M1SL TOTRESNS with 2001-10: 0.675 | without: 0.396
TOTRESNS NONBORRES with 2001-10: 0.994 | without: 0.991


## Method

## Forecasting

## Evaluation

## Results